In [0]:
df = spark.read.table("shopkart.bronze.orders_dirty")

In [0]:
df2 = df.dropDuplicates(["order_id"])

In [0]:
from pyspark.sql import functions as F
df3 = df2.withColumn(
    "order_status", 
    F.lower(F.trim(F.coalesce(F.col("order_status"), F.lit("Unknown"))))
)


In [0]:
df3.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType


timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "_extracted_at"
]

for column_name in timestamp_columns:
    df4 = df3.withColumn(
        column_name,
        F.to_timestamp(F.col(column_name), "yyyy-MM-dd HH:mm:ss")
    )

In [0]:
from pyspark.sql.types import IntegerType,TimestampType

df5 = df4.withColumn(
    "order_purchase_timestamp",
    df4["order_purchase_timestamp"].cast(TimestampType())
).withColumn(
    "order_approved_at",
    df4["order_approved_at"].cast(TimestampType())
).withColumn(
    "order_delivered_carrier_date",
    df4["order_delivered_carrier_date"].cast(TimestampType())
).withColumn(
    "order_delivered_customer_date",
    df4["order_delivered_customer_date"].cast(TimestampType())
).withColumn(
    "order_estimated_delivery_date",
    df4["order_estimated_delivery_date"].cast(TimestampType())
).withColumn(
    "_extracted_at",
    df4["_extracted_at"].cast(TimestampType())
)

df5.printSchema()

In [0]:
df6 = df5.withColumn("delivery_days", df5.order_delivered_customer_date - df5.order_purchase_timestamp)

In [0]:
from pyspark.sql import functions as F
df7 = df6.withColumn("is_valid_delivery", 
                             F.when(F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"),'True')
                                .otherwise("False"))

In [0]:
df7.show()

In [0]:
df7.write.format("delta").mode("overwrite").saveAsTable("shopkart.silver.orders")

In [0]:
# ---- SK-04: "before" profile of bronze.orders_dirty (run this again on silver.orders for "after") ----
from pyspark.sql import functions as F

df = spark.table("shopkart.silver.orders")
total = df.count()
profile = {
    "total rows":                total,
    "distinct order_id":         df.select("order_id").distinct().count(),
    "exact duplicate rows":      total - df.distinct().count(),
    "NULL order_status":         df.filter(F.col("order_status").isNull()).count(),
    "non-normalised status":     df.filter(F.col("order_status") != F.lower(F.trim("order_status"))).count(),
    "delivered before purchase": df.filter(F.to_timestamp("order_delivered_customer_date")
                                           < F.to_timestamp("order_purchase_timestamp")).count(),
}
profile["key duplicate rows (total - distinct ids)"] = total - profile["distinct order_id"]
for k, v in profile.items():
    print(f"{k:<45} {v:>7,}")